Steps to preprocess MODIS data for phenology tracking. These methods are 100% adapted from Jerod Merkle and Shannon Albeke's workflow to process data in R. 

Steps 1a, 1b and the first function in step 2 (ndviCALC_clean) all run within GEE (~10 seconds). However, GEE runs into memory issues when trying to fit the curves on the image collections (the spatial data that is stored in GEE). So, we export the image collections as rasters and then will read it back in.

Conversion to Python and GEE by Tayler LaSharr

October 2024

In [11]:
## first we will read in necessary python libraries and intialize the GEE module
## you will have to authenticate GEE on your machine to use this code

import ee
import matplotlib.pyplot as plt
from datetime import datetime, timezone
import geopandas as gpd
from shapely.geometry import box
import pandas as pd
import geemap
import numpy as np
from scipy.optimize import curve_fit # for fitting the DLC curves (similar to NLS in R)
import multiprocessing as mp


## if you need to authenticate GEE on your machine, use this code (will give you a link to allow you to authenticate it online )
# ee.Authenticate()

## initialize the Earth Engine module (change 'tlasharr' to your project name)
ee.Initialize(project='ee-tlasharr')

Step 1a. Obtain Data.

This step reads in the different data pieces that we care about (modis NDVI, MODIS snow, and mule deer polygons) and define the time intervals that you want to classify winter as anbd the time series for the data you are interested in. This differs very slighly from the R code and uses the day-months instead of the Julian date. This difference is solely to help deal with leap years in systems where winter may end before the end of Feburary when I am working in desert systems.

In [12]:
## set thresholds and things to manipulate

## define the winter period for your study area
start_winter_month_day = '11-01'
end_winter_month_day = '03-01'

## get the time period we care about
start_date_data = '2001-01-01'
end_date_data = '2012-12-31'

## load in a shapefile or area of interest, this is an assest that is already loaded into GEE
# md_cooridor = ee.FeatureCollection('projects/ee-tlasharr/assets/WY_Sublette') ## "normal" temperate environment
# md_cooridor = ee.FeatureCollection('projects/ee-tlasharr/assets/UT_Paunsaugunt') ## some snow in winter
# md_cooridor = ee.FeatureCollection('projects/ee-tlasharr/assets/AZ_RainbowValley') ## some snow in winter

## get a bounding box for clipping purposes
# bounding_box = md_cooridor.geometry().bounds()

## alternatively, you can create a bounding box from coordinates.
# bounding_box = ee.Geometry.Rectangle([-112.125449, 40.010084, -102.721152, 45.717045])
bounding_box = ee.Geometry.Rectangle([-112.125449, 44.010084, -111.721152, 45.717045])

## load the MODIS NDVI data
modis_ndvi = ee.ImageCollection('MODIS/061/MOD09Q1') \
                .filterDate(start_date_data, end_date_data) \
                .select('sur_refl_b01', 'sur_refl_b02', 'QA', 'State')

## calculate NDVI
def calc_ndvi(image):
    ndvi = image.expression(
        '((NIR - RED) / (NIR + RED))', {
            'NIR': image.select('sur_refl_b02'),
            'RED': image.select('sur_refl_b01')
        }).rename('NDVI')
    return image.addBands(ndvi)

## add the NDVI band to the modis collection
modis_ndvi = modis_ndvi.map(calc_ndvi)

## get the projection of the MODIS NDVI data to reproject the bounding box
modis_projection = modis_ndvi.first().projection()
modis_scale = modis_projection.nominalScale()

## reproject the bounding box to the same projection as MODIS
reprojected_bounding_box = bounding_box.transform(modis_projection, modis_scale)

## clip the modis data to the bounding box
modis_ndvi = modis_ndvi.map(lambda image: image.clip(reprojected_bounding_box))

## make sure the time property is there and correct (i.e., dates for the images)
def check_time_start(image):
    return ee.Algorithms.If(
        image.get('system:time_start'),
        image,
        image.set('system:time_start', ee.Date(image.date()).millis())
    )
modis_ndvi = modis_ndvi.map(lambda img: ee.Image(check_time_start(img)))


Step 1b. Get an image collection of the filtered NDVI data and an image collection of the snow cover data.

First function: removes low-quality observations with the QA band. This function will create an image collection called 'NDVIFiltered' that has filtered NDVI data during the time period you are interested in (i.e., removes poor quality pixels, snow, and clouds). This is the equivalent of the 'NDVI' rasters produced in step 1b of the R code.

Second function: creates a Snow Free Mask. This function will create an image collection called 'SnowCover' that has an image with pixels set to 1 (snow) or 0 (no snow) for each image during the time period you are interested in. This is the equivalent of the 'SnowCover' rasters produced in step 1b of the R code.

In [13]:
## this function removes poor quality pixels, pixels with cloud cover, and pixels with snow
def quality_and_state_mask(image):

    ## get both the state and quality bands
    state = image.select('State')
    quality = image.select('QA')

    ## state band masks
    good_cloud_state = state.bitwiseAnd(3).lt(2)  ## clear or cloudy, exclude mixed and not set
    no_cloud_shadow = state.bitwiseAnd(1 << 2).eq(0)  ## no cloud shadow
    no_snow_ice = state.bitwiseAnd(1 << 12).eq(0)  ## no snow/ice

    ## quality band masks
    good_quality = quality.bitwiseAnd(3).eq(0)  ## highest quality

    ## combine masks: apply all conditions
    combined_mask = good_cloud_state.And(no_cloud_shadow).And(good_quality).And(no_snow_ice)

    ## apply the mask to the NDVI band specifically
    ndvi_masked = image.updateMask(combined_mask).select('NDVI')

    ## then, get rid of outliers
    ndvi_masked =  ndvi_masked.updateMask(ndvi_masked.gte(-1).And(ndvi_masked.lte(2)))

    ## make sure that it stays in the native MODIS projection
    ndvi_masked = ndvi_masked.reproject(image.projection())

    ## return the masked image
    return ndvi_masked

## this function is to extract snow information from the State band of the MODIS data
def extract_snow(image):
    
    ## get the state band
    state = image.select('State')

    ## bit 12 represents snow/ice
    snow_mod35 = state.bitwiseAnd(1 << 12).neq(0)  # 1 if snow/ice is present, 0 otherwise

    ## add the snow band to the image
    snow_band = snow_mod35.rename('Snow')

    ## carry over the system:time_start property from the original image (this is not saved in the State band, so you need to explicitly call it back in)
    ## and make sure it stays in the native projection

    snow_band = snow_band.reproject(image.projection()).set('system:time_start', image.get('system:time_start'))

    return snow_band


## apply the quality mask to the MODIS collection
NDVIFiltered = modis_ndvi.map(lambda img: quality_and_state_mask(img))

## apply the snow extraction function to the MODIS collection
SnowCover = modis_ndvi.map(extract_snow)


Step 2. Calculations - the following functions go through 7/8 of the main steps in the ndviCALC_clean function from R. I was running into memory issues with GEE when I tried scaling the data, so I exported after step 7 and will read back in the data as a raster to finish the remainder of the steps.

ndviCALC_clean:

Perform all steps in the ndviCALC_clean function on the filtered MODIS data, with the exception of scaling the data. I ran into a lot of issues trying to scale the data in GEE, so resorted to exporting it and reading it back in as a GeoTiff/raster in Python.

In [23]:
## step 1: reclassify outliers
def ndviCALC_step1(NDVICollection):
    
    ## reclassify outliers (mask NDVI values less than 0 and greater than 2)
    def reclassify_outliers(image):
        return image.updateMask(image.gte(0).And(image.lte(2)))
    

    NDVICollection = NDVICollection.map(reclassify_outliers)
    return NDVICollection

## step 2: reclassify values during winter as 2
def ndviCALC_step2(NDVICollection, start_winter_month_day='11-01', end_winter_month_day='03-30'):
    ## split the winter start and end dates into months and days
    start_winter_month, start_winter_day = map(int, start_winter_month_day.split('-'))
    end_winter_month, end_winter_day = map(int, end_winter_month_day.split('-'))

    ## function to classify whether the date is in the winter period
    def classify_winter(image):
        ## extract the date of the image
        date = ee.Date(image.get('system:time_start'))
        year = date.get('year')
        
        ## define winter start and end dates for that year
        winter_start = ee.Date.fromYMD(year, start_winter_month, start_winter_day)
        winter_end = ee.Date.fromYMD(year, end_winter_month, end_winter_day)

        ## use date difference to identify if the image falls within winter
        is_in_winter = date.difference(winter_start, 'day').gte(0).And(date.difference(winter_end, 'day').lte(0))

        ## mask the image during winter (i.e., replace NDVI values with 2 during winter)
        image_winter_adjusted = image.where(is_in_winter, ee.Image.constant(2))
        image_winter_adjusted = image_winter_adjusted.reproject(NDVICollection.first().projection())

        return image_winter_adjusted

    ## map the function over the collection
    NDVICollection_winter_adjusted = NDVICollection.map(classify_winter)
    return NDVICollection_winter_adjusted

## step 3: check if pixel has been snow-free for 3 consecutive images
def ndviCALC_step3(NDVICollection, SnowCollection):
    def snow_free_for_3(image, prev_image1, prev_image2):
        ## get current and previous snow cover (unmask to avoid gaps)
        current_snow = image.unmask(1)
        prev_snow1 = prev_image1.unmask(1)
        prev_snow2 = prev_image2.unmask(1)

        ## check if the pixel is snow-free for 3 consecutive images (snow-free if sum == 0)
        snow_free_3_consec = current_snow.add(prev_snow1).add(prev_snow2).eq(0)
        return snow_free_3_consec

    ## convert both the NDVI and Snow collections to lists of images
    ndvi_images = NDVICollection.toList(NDVICollection.size())
    snow_images = SnowCollection.toList(SnowCollection.size())

    ## initialize the collection with the first two images (can't apply the snow-free logic to these)
    snow_free_collection = ee.ImageCollection([ndvi_images.get(0), ndvi_images.get(1)])

    ## loop through the rest of the images (starting from the third)
    for i in range(2, NDVICollection.size().getInfo()):
        current_image_ndvi = ee.Image(ndvi_images.get(i)).reproject(NDVICollection.first().projection())
        current_image_snow = ee.Image(snow_images.get(i)).reproject(SnowCollection.first().projection())
        prev_image1_snow = ee.Image(snow_images.get(i - 1)).reproject(SnowCollection.first().projection())
        prev_image2_snow = ee.Image(snow_images.get(i - 2)).reproject(SnowCollection.first().projection())

        ## apply the snow-free logic to the snow images
        snow_free_image = snow_free_for_3(current_image_snow, prev_image1_snow, prev_image2_snow)

        ## mask the NDVI image based on the snow-free condition
        ndvi_snow_free_masked = current_image_ndvi.updateMask(snow_free_image)

        ## append the result to the collection
        snow_free_collection = snow_free_collection.merge(ee.ImageCollection([ndvi_snow_free_masked]))

    return snow_free_collection

## step 4: mask pixels that are in winter or covered by snow
def ndviCALC_step4(NDVICollection, SnowCollection):

    def mask_winter_snow(ndvi_image):
        ## get the corresponding snow image for the same date
        snow_image = SnowCollection.filterDate(ndvi_image.date(), ndvi_image.date()).first()

        ## ensure that snow_image exists, if not, create an empty image with value 0
        snow_image = ee.Image(ee.Algorithms.If(snow_image, snow_image, ee.Image.constant(0).rename('Snow')))

        ## mask pixels where snow cover is equal to 2 (as set in earlier steps) or during winter
        masked_image = ndvi_image.updateMask(snow_image.neq(2))  # mask where Snow equals 2 (snow-covered)
        masked_image = masked_image.reproject(NDVICollection.first().projection())

        return masked_image

    ## apply the snow and winter mask to the NDVIFiltered collection
    NDVIFiltered_masked = NDVICollection.map(mask_winter_snow)
    return NDVIFiltered_masked

## step 5: apply median filter
def ndviCALC_step5(NDVICollection, window_size=3):
    ## ensure that the NDVI is in float format for precision
    NDVICollection = NDVICollection.map(lambda img: img.select('NDVI').toFloat()) 

    def smooth_image(current_image):
        all_images = NDVICollection.sort('system:time_start')
        all_images_list = all_images.toList(all_images.size())

        image_index = all_images_list.indexOf(current_image)
        half_window = ee.Number(window_size // 2)

        ## calculate the start and end indices for the temporal window
        start_index = ee.Number(image_index).subtract(half_window).max(0)
        end_index = ee.Number(image_index).add(half_window).min(all_images.size().subtract(1))

        ## get the window of images and create a new image collection
        window_images = all_images_list.slice(start_index, end_index.add(1))
        window_collection = ee.ImageCollection.fromImages(window_images)

        ## compute the median NDVI for the current window
        median_image = window_collection.median().toFloat()
        median_image = median_image.reproject(NDVICollection.first().projection())

        ## copy the properties of the current image and return the median NDVI
        return median_image.copyProperties(current_image, current_image.propertyNames())

    # apply the temporal smoothing function to the entire collection
    smoothed_collection = NDVICollection.map(smooth_image)

    return smoothed_collection

## step 6: calculate the floor (0.025 quantile)
def ndviCALC_step6(NDVICollection, quantile_value=2.5):

    ## calculate the 0.025 quantile for each pixel across the entire collection
    baseline_quantile = NDVICollection.reduce(ee.Reducer.percentile([quantile_value]))

    ## use this quantile as a floor for the entire baseline image
    floored_baseline_image = baseline_quantile.rename('NDVI_baseline')

    ## make sure it stays in the native projection
    floored_baseline_image = floored_baseline_image.reproject(NDVICollection.first().projection())

    return floored_baseline_image

## step 7. apply baseline flooring and add winter and snow masks
## step 7: apply baseline flooring during winter
def ndviCALC_step7(NDVICollection, floored_image, start_winter_month_day='11-01', end_winter_month_day='03-30'):
    
    ## function to add a winter mask band
    def add_winter_band(ndvi_image):
        date = ee.Date(ndvi_image.get('system:time_start'))
        year = date.get('year')
        start_month, start_day = map(int, start_winter_month_day.split('-'))
        end_month, end_day = map(int, end_winter_month_day.split('-'))

        ## define winter start and end dates for that year
        winter_start = ee.Date.fromYMD(year, start_month, start_day)
        winter_end = ee.Date.fromYMD(year, end_month, end_day)

        ## determine if the current date falls within the winter period
        is_winter = date.difference(winter_start, 'day').gte(0).And(date.difference(winter_end, 'day').lte(0))

        ## create an image with the winter mask (1 if winter, 0 if not)
        winter_band = ee.Image.constant(is_winter).rename('is_winter')
        return ndvi_image.addBands(winter_band)

    ## function to floor NDVI values below the baseline during winter
    def floor_ndvi_in_winter(ndvi_image):
        ndvi = ndvi_image.select('NDVI')
        baseline_ndvi = floored_image.select('NDVI_baseline')
        is_winter = ndvi_image.select('is_winter')

        ## if it's winter, replace NDVI values with the baseline if they are lower
        floored_ndvi = ndvi.where(is_winter.eq(1).And(ndvi.lt(baseline_ndvi)), baseline_ndvi)

        ## add the floored NDVI as a new band
        return ndvi_image.addBands(floored_ndvi.rename('NDVI'), overwrite=True)

    ## apply the functions to the NDVI collection
    NDVICollection = NDVICollection.map(add_winter_band)
    NDVICollection = NDVICollection.map(floor_ndvi_in_winter)
    
    return NDVICollection


## step 8: replace winter values with baseline and apply snow mask outside of winter
def ndviCALC_step8(NDVICollection, floored_image):
    ## function to replace NDVI values in winter with the baseline
    def correct_winter_values(ndvi_image):
        ndvi = ndvi_image.select('NDVI')
        is_winter = ndvi_image.select('is_winter')
        snow_mask = ndvi_image.select('snow_mask')
        baseline_ndvi = floored_image.select('NDVI_baseline')

        ## if it's winter, replace NDVI values with the baseline NDVI
        corrected_ndvi = ndvi.where(is_winter.eq(1), baseline_ndvi.unmask(baseline_ndvi))

        ## if it's not winter and the pixel is snow-covered, mask it (set to empty)
        corrected_ndvi = corrected_ndvi.updateMask(is_winter.eq(0).And(snow_mask.Not()))

        ## return the corrected NDVI image
        return ndvi_image.addBands(corrected_ndvi.rename('NDVI'), overwrite=True).reproject(NDVICollection.first().projection())

    ## apply the correction function to the NDVI collection
    corrected_collection = NDVICollection.map(correct_winter_values)
    return corrected_collection

## here we could write over the image collections, but it doesn't cause any memory issues and I still am not super confident in GEE stuff and how
## overwriting works for image collections, so am not overwriting them yet.

## run all steps sequentially with the provided NDVI and SnowCover collections

NDVIFiltered_reclassified = ndviCALC_step1(NDVIFiltered)
print("Step 1 complete: Removed outliers from data.")

NDVIFiltered_winter_classified = ndviCALC_step2(NDVIFiltered_reclassified, start_winter_month_day='11-01', end_winter_month_day='03-30')
print("Step 2 complete: Reclassified values during winter.")

NDVIFiltered_snow_free = ndviCALC_step3(NDVIFiltered_winter_classified, SnowCover)
print("Step 3 complete: Checked if pixels have been snow-free for 3 consecutive images.")

NDVIFiltered_masked = ndviCALC_step4(NDVIFiltered_snow_free, SnowCover)
print("Step 4 complete: Masked pixels that are in winter or covered by snow.")

NDVIFiltered_temporal_smoothed = ndviCALC_step5(NDVIFiltered_masked, window_size=3)
print("Step 5 complete: Applied moving window with a window size of 3.")

NDVIFiltered_floored = ndviCALC_step6(NDVIFiltered_temporal_smoothed, quantile_value=2.5)
print("Step 6 complete: Calculated the floor (0.025 quantile).")

NDVICollection_with_masks = ndviCALC_step7(NDVIFiltered_temporal_smoothed, NDVIFiltered_floored, start_winter_month_day='11-01', end_winter_month_day='03-30')
print("Step 7 complete: Applied baseline flooring and added winter and snow masks.")

NDVICollection_corrected = ndviCALC_step8(NDVICollection_with_masks, NDVIFiltered_floored)
print("Step 8 complete: Replaced winter values with baseline and applied snow mask outside of winter.")


Step 1 complete: Removed outliers from data.
Step 2 complete: Reclassified values during winter.
Step 3 complete: Checked if pixels have been snow-free for 3 consecutive images.
Step 4 complete: Masked pixels that are in winter or covered by snow.
Step 5 complete: Applied moving window with a window size of 3.
Step 6 complete: Calculated the floor (0.025 quantile).
Step 7 complete: Applied baseline flooring and added winter and snow masks.
Step 8 complete: Replaced winter values with baseline and applied snow mask outside of winter.


Here, we are going to export the image collections and read them back in as rasters.

In [25]:
## function to export each year with specified region and CRS
def export_quarterly_ndvi_as_layers(NDVIImagecollection, start_year, end_year, export_folder, export_region, modis_projection, modis_scale):
    
    quarters = {
        "Q1": ("01-01", "03-31"),
        "Q2": ("04-01", "06-30"),
        "Q3": ("07-01", "09-30"),
        "Q4": ("10-01", "12-31")
    }
    
    for year in range(start_year, end_year + 1):
        for quarter, (start_date, end_date) in quarters.items():
            
            ## filter by quarter
            quarterly_collection = NDVIImagecollection.filterDate(f'{year}-{start_date}', f'{year}-{end_date}')

            ## reproject each image in the collection to the same CRS and scale
            def reproject_image(image):
                return image.reproject(crs=modis_projection, scale=modis_scale)
            
            ## apply the reprojection
            reprojected_collection = quarterly_collection.map(reproject_image)
            
            ## get a list of the band names to save each layer with the correct date
            band_names = reprojected_collection.aggregate_array('system:time_start').map(
                lambda date: ee.Date(date).format('YYYY-MM-dd')
            )

            ## convert the image collection to a single image with each band corresponding to an image in the collection
            def band_names_func(image, prev_image):
                prev_image = ee.Image(prev_image)
                updated = prev_image.addBands(image.select('NDVI'))
                return updated

            ## create an initial image with a dummy band to start adding bands
            dummy_image = reprojected_collection.first().select('NDVI').multiply(0).rename('dummy')
            
            ## accumulate all images as bands
            quarterly_image = ee.Image(reprojected_collection.iterate(band_names_func, dummy_image))
            
            ## remove the initial placeholder dummy band
            band_list = quarterly_image.bandNames().remove('dummy')
            quarterly_image = quarterly_image.select(band_list)
            
            ## rename with the date strings
            quarterly_image = quarterly_image.rename(band_names)

            ## define export task
            export_task = ee.batch.Export.image.toDrive(
                image=quarterly_image,
                description=f'NDVI_{year}_{quarter}',
                folder=export_folder,
                fileNamePrefix=f'NDVI_{year}_{quarter}',
                region=export_region,
                scale=modis_scale,  # Use MODIS native scale (e.g., 250m)
                maxPixels=1e13,
                crs=modis_projection,  # Use MODIS native projection
                fileFormat='GeoTIFF'
            )

            ## start the export task
            export_task.start()
            print(f'Exporting NDVI_{year}_{quarter} to Google Drive in {export_folder}.')

## define parameters for export
export_region = bounding_box
modis_projection = NDVICollection_corrected.first().projection()
modis_scale = 250  # Set to MODIS native resolution, e.g., 500m or 250m

## call the function for each year
export_quarterly_ndvi_as_layers(NDVICollection_with_masks, 2001, 2002, 'NDVI_Exports', export_region, modis_projection, modis_scale)


Exporting NDVI_2001_Q1 to Google Drive in NDVI_Exports.
Exporting NDVI_2001_Q2 to Google Drive in NDVI_Exports.
Exporting NDVI_2001_Q3 to Google Drive in NDVI_Exports.
Exporting NDVI_2001_Q4 to Google Drive in NDVI_Exports.
Exporting NDVI_2002_Q1 to Google Drive in NDVI_Exports.
Exporting NDVI_2002_Q2 to Google Drive in NDVI_Exports.
Exporting NDVI_2002_Q3 to Google Drive in NDVI_Exports.
Exporting NDVI_2002_Q4 to Google Drive in NDVI_Exports.


(ndviCALC_dlc, and ndviCALC_vals). Additionally, it exports the image collections at the end of the first function and reads them back in as rasters.